In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv(f"{path}/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
plt.figure()
plt.hist(df["Delivery_Time"], bins=30)
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Delivery Time")
plt.show()



In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=["Order_ID"])

In [ ]:
# Task 2: Write your code here:
df.isna().sum()
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

In [ ]:
# Task 3: Write your code here:
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:
y.describe()

In [ ]:
# Task 1: Write your code here:
X = X_scaled
y = y.values

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

print("Average MAE across folds:", np.mean(mae_scores))


In [ ]:
# Task 1: Write your code here:
from sklearn.ensemble import RandomForestRegressor
rf_final = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
rf_final.fit(X_scaled, y)

import matplotlib.pyplot as plt
import numpy as np

importances = rf_final.feature_importances_
feature_names = df.drop(columns=["Delivery_Time"]).columns
indices = np.argsort(importances)
plt.figure(figsize=(8, 6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), feature_names[indices])
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importance")
plt.show()



In [ ]:
# Task 2: Write your code here:
y_pred_full = rf_final.predict(X_scaled)

plt.figure()
plt.hist(y_pred_full, bins=30)
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Predicted Delivery Time")
plt.show()

In [ ]:
pip install catboost

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

ensemble_mae_scores = []

for train_idx, val_idx in kf.split(X_scaled):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Model 1: Random Forest
    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_val)

    # Model 2: CatBoost
    cb = CatBoostRegressor(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        verbose=False,
        random_state=42
    )
    cb.fit(X_train, y_train)
    cb_preds = cb.predict(X_val)

    # Average predictions
    avg_preds = (rf_preds + cb_preds) / 2

    # MAE on averaged predictions
    mae = mean_absolute_error(y_val, avg_preds)
    ensemble_mae_scores.append(mae)


In [ ]:
print("Average Ensemble MAE:", np.mean(ensemble_mae_scores))